# Issues 1 & 2 — TF-expression misalignment in the force-curve calculation

Companion to `tests/ISSUES.md`.  Everything here runs against the miniature but
**real** `dictys.net.dynamic_network` (built inline, section 0.1), and calls the
production functions directly — nothing is stubbed or monkeypatched except where
a proposed fix is demonstrated (clearly marked).

| | issue | status in `ISSUES.md` | what this notebook shows |
|---|---|---|---|
| **1** | `calculate_force_curves_chunk` mis-assigns TF expression | **CONFIRMED, high** | the full `EpisodeDynamics` pipeline producing wrong `avg_force` for real edges |
| **2** | `SmoothedCurvesGRN.calculate_force_curves` is positionally coupled | **REVISED, medium** | why the cross product from `get_beta_curves` normally saves it, and the ordering footgun that remains |

**Run with the `dictys` kernel.**  Top-to-bottom, ~30 s.

## 0. Setup

In [1]:
# Self-contained: nothing is imported from the test suite and nothing depends on
# the working directory.  The only requirement is that `dictys` and `firefate`
# are importable -- i.e. that this runs on the `dictys` kernel.
import os, subprocess, sys, textwrap, inspect

import numpy as np
import pandas as pd
import dictys
import dictys.traj

from firefate.core.episodic_dynamics import (
    EpisodeDynamics,
    calculate_force_curves_chunk,
    calculate_force_curves_parallel,
    create_balanced_chunks,
    filter_chunk_of_edges,
    filter_edges_by_significance_and_direction,
)
from firefate.core.pseudotime_curves import SmoothedCurvesGRN
from firefate.utils.custom import get_gene_indices, get_tf_indices

pd.set_option("display.width", 120)
print("pandas", pd.__version__, "| numpy", np.__version__)
print("firefate:", os.path.dirname(inspect.getfile(SmoothedCurvesGRN)))

pandas 2.2.2 | numpy 2.0.2
firefate: /projects/bhdw/afu2/FIREFate/src/firefate/core


In [ ]:
EPS = 1e-10
LOG10E = 1.0 / np.log(10.0)          # exp(log10 a + log10 b) = (a*b) ** (1/ln 10)

def reference_force(beta, tf_expr, eps=EPS):
    """The transform the library evaluates, with the CORRECT pairing of TF expression.

    sign(b) * exp(log10(|b| + eps) + log10(t + eps))
    """
    return np.sign(beta) * ((np.abs(beta) + eps) * (tf_expr + eps)) ** LOG10E

def implied_expression(force, beta, eps=EPS):
    """Invert the transform to recover which expression value a force was built from.

    Lets us *name* the TF whose expression each edge actually received.
    """
    return np.abs(force) ** np.log(10.0) / (np.abs(beta) + eps) - eps

# sanity check on the inversion itself
_b, _t = np.array([2.0, -1.5]), np.array([3.0, 7.0])
assert np.allclose(implied_expression(reference_force(_b, _t), _b), _t)
print("reference_force / implied_expression agree")

reference_force / implied_expression agree


### 0.1 The mock network

A miniature but **real** `dictys.net.dynamic_network` -- the same class the
pipeline takes in production, so nothing is stubbed.  Same construction as
`tests/conftest.py`, inlined here so the notebook stands alone and does not care
where the kernel was started.  The edge dictionary is a parameter, which is what
section 2 varies.

```
trajectory:  node0 --e0-- node1 --e1-- node2        (each edge length 1)
                             \--e2-- node3
windows:     0(n0) 1 2(n1) 3 4(n2) 5 6(n3)          3 cells sitting on each
genes:       TFA TFB ZNF1 G1 G2 G3 G4 G5            regulators: TFA, TFB, ZNF1
```

CPM values are chosen so `log2(CPM + 1)` is a round number: **TFA is flat at
2.0** while **TFB rises 1 -> 4**, so a swap between the two is visible at every
pseudotime point.

In [3]:
N_WINDOWS  = 7
GENES      = np.array(["TFA", "TFB", "ZNF1", "G1", "G2", "G3", "G4", "G5"])
GIDX       = {g: i for i, g in enumerate(GENES)}
REGULATORS = ["TFA", "TFB", "ZNF1"]

CONSTANT_LCPM = {"TFA": 2.0, "ZNF1": 1.0, "G1": 4.0, "G5": 0.0}   # flat curves
RISING_LCPM   = ["TFB", "G2"]                                     # rise 1 -> 4

def _cpm_matrix():
    cpm = np.zeros((len(GENES), N_WINDOWS))
    for gene, lcpm in CONSTANT_LCPM.items():
        cpm[GIDX[gene]] = 2.0 ** lcpm - 1.0
    for gene in RISING_LCPM:
        cpm[GIDX[gene]] = 2.0 ** np.linspace(1.0, 4.0, N_WINDOWS) - 1.0
    cpm[GIDX["G3"]] = 2.0 ** np.linspace(4.0, 1.0, N_WINDOWS) - 1.0
    cpm[GIDX["G4"]] = 2.0 ** np.array([1.0, 2.0, 3.0, 4.0, 3.0, 2.0, 1.0]) - 1.0
    return cpm

def build_mock_network(edges):
    """edges: {(tf, target): per-window weight array}; every other edge is 0."""
    traj = dictys.traj.trajectory(np.array([[0, 1], [1, 2], [1, 3]]),
                                  np.array([1.0, 1.0, 1.0]))
    w_edge = np.array([0, 0, 0, 1, 1, 2, 2])
    w_loc  = np.array([0.0, 0.5, 1.0, 0.5, 1.0, 0.5, 1.0])
    pts_s  = dictys.traj.point(traj, w_edge.copy(), w_loc.copy())
    pts_c  = dictys.traj.point(traj, np.repeat(w_edge, 3), np.repeat(w_loc, 3))

    sc_w = np.zeros((N_WINDOWS, N_WINDOWS * 3))
    for i in range(N_WINDOWS):
        sc_w[i, i * 3:(i + 1) * 3] = 1

    neighbours = np.zeros((N_WINDOWS, N_WINDOWS), dtype=int)
    for a, b in [(0, 1), (1, 2), (2, 3), (3, 4), (2, 5), (5, 6)]:
        neighbours[a, b] = neighbours[b, a] = 1

    net = np.zeros((len(REGULATORS), len(GENES), N_WINDOWS))
    for (tf, target), vals in edges.items():
        net[REGULATORS.index(tf), GIDX[target]] = vals

    return dictys.net.dynamic_network(
        cname=np.array([f"cell{i}" for i in range(N_WINDOWS * 3)]),
        sname=np.array([f"Subset{i + 1}" for i in range(N_WINDOWS)]),
        nname=GENES.copy(),
        nids=[np.array([GIDX[t] for t in REGULATORS]), np.arange(len(GENES))],
        traj=traj, point={"c": pts_c, "s": pts_s},
        scprop={"w": sc_w}, ssprop={"traj-neighbor": neighbours},
        nsprop={"cpm": _cpm_matrix()},
        esprop={"w": net, "w_n": net * 0.5, "w_in": net * 2.0, "mask": net != 0},
    )

STANDARD_EDGES = {                       # the seven edges of the shared fixture
    ("TFA", "G1"):  np.full(N_WINDOWS, 2.0),
    ("TFA", "G2"):  np.linspace(1.0, 2.0, N_WINDOWS),
    ("TFA", "G3"):  np.linspace(3.0, -3.0, N_WINDOWS),
    ("TFA", "TFB"): np.full(N_WINDOWS, 1.0),
    ("TFB", "G4"):  np.full(N_WINDOWS, -1.5),
    ("TFB", "G5"):  np.linspace(-0.5, -1.5, N_WINDOWS),
    ("ZNF1", "G1"): np.full(N_WINDOWS, 5.0),
}

_check = build_mock_network(STANDARD_EDGES)
print("built", type(_check).__name__, "with genes", _check.nname.tolist())

built dynamic_network with genes ['TFA', 'TFB', 'ZNF1', 'G1', 'G2', 'G3', 'G4', 'G5']


### 0.2 Provenance of the call sequence

Section 2 does not invent an order — it is the sequence FIREFate itself runs, in
`EnrichmentManager.enrich_episodic` and `run_episodic_construction`.  Printed here so
the two can be compared directly.

In [4]:
import re
from firefate.managers.enrichment_manager import EnrichmentManager
from firefate.core.episodic_dynamics import run_episodic_construction

def pipeline_calls(fn):
    """Every epi.<method>() call a production entry point makes, in order.

    Regex rather than a startswith("epi.") scan: some calls are assigned, e.g.
    ``avg_force = epi.calculate_forces()``.
    """
    return re.findall(r"\bepi\.(\w+)\s*\(", inspect.getsource(fn))

mgr = pipeline_calls(EnrichmentManager.enrich_episodic)
run = pipeline_calls(run_episodic_construction)
NOTEBOOK_SEQUENCE = ["compute_expression_curves", "build_episode_grn", "filter_edges",
                     "compute_tf_expression", "calculate_forces", "select_top_edges"]

print("EnrichmentManager.enrich_episodic:", mgr)
print("run_episodic_construction        :", run)
print("this notebook, section 2         :", NOTEBOOK_SEQUENCE)
print()
print("the notebook runs the production order, minus the LF steps that follow the forces:")
print("   subsequence of enrich_episodic          :",
      [c for c in mgr if c in NOTEBOOK_SEQUENCE] == NOTEBOOK_SEQUENCE)
print("   identical to run_episodic_construction  :", run == NOTEBOOK_SEQUENCE)

EnrichmentManager.enrich_episodic: ['compute_expression_curves', 'set_lf_genes', 'build_episode_grn', 'filter_edges', 'compute_tf_expression', 'calculate_forces', 'select_top_edges', 'annotate_lf_in_grn', 'calculate_enrichment']
run_episodic_construction        : ['compute_expression_curves', 'build_episode_grn', 'filter_edges', 'compute_tf_expression', 'calculate_forces', 'select_top_edges']
this notebook, section 2         : ['compute_expression_curves', 'build_episode_grn', 'filter_edges', 'compute_tf_expression', 'calculate_forces', 'select_top_edges']

the notebook runs the production order, minus the LF steps that follow the forces:
   subsequence of enrich_episodic          : True
   identical to run_episodic_construction  : True


## 1. The mechanism, shared by both issues

Three lines inside `calculate_force_curves_chunk`.  Read the *indices* they use:

In [5]:
src = inspect.getsource(calculate_force_curves_chunk)
start = src.index("    # Get unique TFs")
print(src[start:src.index("    # Convert to numpy")])

    # Get unique TFs in this chunk
    tfs_in_chunk = beta_chunk.index.get_level_values(0).unique()

    # Count number of targets per TF in this chunk
    targets_per_tf = beta_chunk.index.get_level_values(0).value_counts()

    # Get TF expression data for TFs in this chunk, in the same order as targets_per_tf
    tf_expr_subset = tf_expression.loc[targets_per_tf.index]

    # Create expanded TF expression DataFrame to match beta_chunk structure
    expanded_tf_expr = pd.DataFrame(
        np.repeat(tf_expr_subset.values, targets_per_tf.values, axis=0),
        index=beta_chunk.index,
        columns=beta_chunk.columns,
    )




* `targets_per_tf` is **count-ordered** (`value_counts()` sorts by descending count).
* `tf_expr_subset` is therefore in count order, and `np.repeat` emits its rows as
  contiguous blocks **in that same count order**.
* those blocks are then glued onto `beta_chunk.index`, which is in **row order**.

The two coincide only when the beta rows happen to be grouped by descending
target count.  So everything turns on how `value_counts()` orders things:

In [6]:
demo = pd.DataFrame({
    "index given to value_counts": [
        "['TFA','TFB','TFB']  (unequal counts)",
        "['TFB','TFB','TFA']  (unequal counts)",
        "['TFA','TFA','TFB','TFB']  (TIED counts)",
        "['TFB','TFB','TFA','TFA']  (TIED counts)",
    ],
    "value_counts order": [
        list(pd.Index(["TFA", "TFB", "TFB"]).value_counts().index),
        list(pd.Index(["TFB", "TFB", "TFA"]).value_counts().index),
        list(pd.Index(["TFA", "TFA", "TFB", "TFB"]).value_counts().index),
        list(pd.Index(["TFB", "TFB", "TFA", "TFA"]).value_counts().index),
    ],
})
demo["matches row order?"] = ["NO", "yes", "yes", "yes"]
demo

,index given to value_counts,value_counts order,matches row order?
0,"['TFA','TFB','TFB'] (unequal counts)","[TFB, TFA]",NO
1,"['TFB','TFB','TFA'] (unequal counts)","[TFB, TFA]",yes
2,"['TFA','TFA','TFB','TFB'] (TIED counts)","[TFA, TFB]",yes
3,"['TFB','TFB','TFA','TFA'] (TIED counts)","[TFB, TFA]",yes


**Unequal counts → re-sorted → can disagree with row order.  Tied counts →
first-appearance order → always agrees.**

That single fact is the whole difference between issue 1 and issue 2:

* the **episodic** path (issue 1) filters edges, so counts end up unequal;
* `get_beta_curves` (issue 2) returns a full cross product, so counts are equal.

Note the tie behaviour is not part of the pandas contract — `value_counts()` is
documented to sort by count, and the ordering of ties is unspecified.

---
## 2. Issue 1 — the episodic pipeline produces wrong forces

### 2.1 A network where the two orders disagree

The shared test fixture accidentally hides this: there, TFA has both
*more* targets **and** a lower `nids[0]` index, so count order and row order
agree.  Below, TFA regulates one gene and TFB regulates three — the ordinary
situation once `filter_edges` has thinned the GRN.

In [7]:
# TFA regulates one gene, TFB three -- the ordinary shape once filter_edges has
# thinned the GRN, and the one the shared test fixture happens to avoid.
SKEWED_EDGES = {
    ("TFA", "G1"): np.full(N_WINDOWS,  2.0),    # TFA: 1 target
    ("TFB", "G2"): np.full(N_WINDOWS, -1.5),    # TFB: 3 targets
    ("TFB", "G3"): np.full(N_WINDOWS,  0.8),
    ("TFB", "G4"): np.full(N_WINDOWS, -2.5),
}

net_skewed   = build_mock_network(SKEWED_EDGES)
net_standard = build_mock_network(STANDARD_EDGES)   # used in section 3

print("regulators, in nids[0] order:",
      [net_skewed.nname[i] for i in net_skewed.nids[0]])
pd.DataFrame({f"{tf}->{tg}": v for (tf, tg), v in SKEWED_EDGES.items()},
             index=[f"window{i}" for i in range(N_WINDOWS)]).T

regulators, in nids[0] order: [np.str_('TFA'), np.str_('TFB'), np.str_('ZNF1')]


,window0,window1,window2,window3,window4,window5,window6
TFA->G1,2.0,2.0,2.0,2.0,2.0,2.0,2.0
TFB->G2,-1.5,-1.5,-1.5,-1.5,-1.5,-1.5,-1.5
TFB->G3,0.8,0.8,0.8,0.8,0.8,0.8,0.8
TFB->G4,-2.5,-2.5,-2.5,-2.5,-2.5,-2.5,-2.5


The regulator expression matters too: **TFA's log2 CPM is flat at 2.0 while
TFB's rises from 1 to 4**, so swapping them is visible at every time point.

In [8]:
epi = EpisodeDynamics(
    net_skewed, output_folder="/tmp", mode="expression",
    trajectory_range=(0, 2), num_points=10, dist=0.3, sparsity=0.1,
)
lcpm, dtime = epi.compute_expression_curves()          # -> SmoothedCurvesGRN.get_smoothed_curves
print("pseudotime:", np.round(dtime.values, 3))
lcpm.loc[["TFA", "TFB"]].round(4)

pseudotime: [0.    0.222 0.444 0.667 0.889 1.111 1.333 1.556 1.778 2.   ]


,0,1,2,3,4,5,6,7,8,9
TFA,2.0000,2.0000,2.0000,2.0000,2.0000,2.0000,2.0000,2.0000,2.0000,2.0000
TFB,1.1026,1.2493,1.4501,1.6655,1.8907,2.1093,2.3345,2.5499,2.7507,2.8974


### 2.2 `build_episode_grn` — rows are grouped in network order

In [9]:
grn = epi.build_episode_grn(time_slice=slice(0, 5))
print("row order (nids[0] order, TFA first):")
grn.round(4)

row order (nids[0] order, TFA first):


time_0  time_1  time_2  time_3  time_4
TF  Target                                        
TFA G1         2.0     2.0     2.0     2.0     2.0
TFB G2        -1.5    -1.5    -1.5    -1.5    -1.5
    G3         0.8     0.8     0.8     0.8     0.8
    G4        -2.5    -2.5    -2.5    -2.5    -2.5

### 2.3 `filter_edges` — this is where the counts become unequal

In [10]:
kept = epi.filter_edges(n_processes=2, chunk_size=100)   # -> filter_edges_by_significance_and_direction
                                                         #    -> filter_chunk_of_edges
kept.round(4)

Processing 4 rows using 2 processes...
Chunk size: 100 rows
Direction invariance check: Enabled
Time columns: ['time_0', 'time_1', 'time_2', 'time_3', 'time_4']
Creating index chunks...
Created 1 chunks of indices


Processing chunks:   0%|                                                                                                                  | 0/1 [00:00<?, ?it/s]/u/afu2/.conda/envs/dictys/lib/python3.9/site-packages/scipy/stats/_axis_nan_policy.py:531: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
Processing chunks: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 44.29it/s]

Processing completed in 0.11 seconds
Sorting results...
Creating result DataFrame...


time_0  time_1  time_2  time_3  time_4  p_value
TF  Target                                                 
TFA G1         2.0     2.0     2.0     2.0     2.0      0.0
TFB G2        -1.5    -1.5    -1.5    -1.5    -1.5      0.0
    G3         0.8     0.8     0.8     0.8     0.8      0.0
    G4        -2.5    -2.5    -2.5    -2.5    -2.5      0.0

In [11]:
row_order   = list(dict.fromkeys(kept.index.get_level_values(0)))
count_order = list(kept.index.get_level_values(0).value_counts().index)
counts      = dict(kept.index.get_level_values(0).value_counts())

print("targets per TF      :", {str(k): int(v) for k, v in counts.items()})
print("row order (as built):", [str(t) for t in row_order])
print("value_counts order  :", [str(t) for t in count_order])
print()
print("DISAGREE ->" if row_order != count_order else "agree ->",
      "the positional np.repeat will attach the wrong blocks"
      if row_order != count_order else "no visible error here")

targets per TF      : {'TFB': 3, 'TFA': 1}
row order (as built): ['TFA', 'TFB']
value_counts order  : ['TFB', 'TFA']

DISAGREE -> the positional np.repeat will attach the wrong blocks


### 2.4 `compute_tf_expression` — the frame that is about to be mis-paired

In [12]:
tf_expr = epi.compute_tf_expression()
tf_expr.round(4)

,time_0,time_1,time_2,time_3,time_4
TF,,,,,
TFA,2.0000,2.0000,2.0000,2.0000,2.0000
TFB,1.1026,1.2493,1.4501,1.6655,1.8907


### 2.5 `calculate_forces` — the wrong numbers appear

In [13]:
avg = epi.calculate_forces(n_processes=2, chunk_size=100)   # -> calculate_force_curves_parallel
                                                            #    -> create_balanced_chunks
                                                            #    -> calculate_force_curves_chunk
beta = kept.drop("p_value", axis=1)

rows = []
for tf, target in beta.index:
    b    = beta.loc[(tf, target)].values
    want = reference_force(b, tf_expr.loc[tf].values).mean()
    got  = avg.loc[(tf, target), "avg_force"]
    rows.append({
        "edge": f"{tf}->{target}",
        "avg_force (library)": got,
        "avg_force (correct)": want,
        "rel. error %": 100 * abs(got - want) / abs(want),
        "verdict": "ok" if abs(got - want) < 1e-9 else "WRONG",
    })
verdict = pd.DataFrame(rows)
verdict.round(4)

Processing 4 edges using 2 processes...
Chunk size: 100 rows
Time columns: ['time_0', 'time_1', 'time_2', 'time_3', 'time_4']
Beta curves shape: (4, 5)
TF expression shape: (2, 5)
Created 1 chunks
Chunk sizes: [4]...


Processing chunks: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 144.79it/s]

Processing completed in 0.07 seconds
Combining results...
Final shape: (4, 5)


,edge,avg_force (library),avg_force (correct),rel. error %,verdict
0,TFA->G1,1.5909,1.8259,12.8713,WRONG
1,TFB->G2,-1.4040,-1.4040,0.0000,ok
2,TFB->G3,1.0686,1.0686,0.0000,ok
3,TFB->G4,-2.0117,-1.7528,14.7727,WRONG


Two of the four edges are wrong, and nothing warned about it.  Naming the
culprit: invert the transform to recover the expression value each edge was
actually multiplied by, and match it against the two regulators' curves.

In [14]:
force_curves = epi.force_curves
rows = []
for tf, target in beta.index:
    b   = beta.loc[(tf, target)].values
    imp = implied_expression(force_curves.loc[(tf, target)].values, b)
    used = next((cand for cand in tf_expr.index
                 if np.allclose(imp, tf_expr.loc[cand].values, atol=1e-6)), "?")
    rows.append({"edge": f"{tf}->{target}", "expression it should use": tf,
                 "expression it actually used": used,
                 "implied values": np.round(imp, 3)})
pd.DataFrame(rows)

,edge,expression it should use,expression it actually used,implied values
0,TFA->G1,TFA,TFB,"[1.103, 1.249, 1.45, 1.665, 1.891]"
1,TFB->G2,TFB,TFB,"[1.103, 1.249, 1.45, 1.665, 1.891]"
2,TFB->G3,TFB,TFB,"[1.103, 1.249, 1.45, 1.665, 1.891]"
3,TFB->G4,TFB,TFA,"[2.0, 2.0, 2.0, 2.0, 2.0]"


`TFA->G1` was scaled by **TFB's** expression and `TFB->G4` by **TFA's** — exactly
the block swap predicted in §1: expression blocks arrive as
`[TFB, TFB, TFB, TFA]` while the rows are `[TFA, TFB, TFB, TFB]`.

### 2.6 The same thing in isolation

Straight into `calculate_force_curves_chunk`, no pipeline, to rule out anything
upstream:

In [15]:
mini_index = pd.MultiIndex.from_tuples(
    [("TFA", "G1"), ("TFB", "G2"), ("TFB", "G3")], names=["TF", "Target"])
mini_beta = pd.DataFrame([[1.0], [1.0], [1.0]], index=mini_index, columns=["time_0"])
mini_expr = pd.DataFrame([[10.0], [1000.0]], index=["TFA", "TFB"], columns=["time_0"])

out = calculate_force_curves_chunk(mini_beta, mini_expr)
pd.DataFrame({
    "force": out["time_0"],
    "implied expression": implied_expression(out["time_0"].values, mini_beta["time_0"].values).round(3),
    "should be": [10.0, 1000.0, 1000.0],
})

force  implied expression  should be
TF  Target                                          
TFA G1      20.085537              1000.0       10.0
TFB G2      20.085537              1000.0     1000.0
    G3       2.718282                10.0     1000.0

### 2.7 Downstream: the ranking, and what gets selected

`select_top_edges` ranks by `|avg_force|`, so mis-scaling reorders the ranking —
and at a tight enough cut it changes *which* edges survive.

In [16]:
correct_avg = pd.Series(
    {idx: reference_force(beta.loc[idx].values, tf_expr.loc[idx[0]].values).mean()
     for idx in beta.index}, name="avg_force").to_frame()

library_rank = avg["avg_force"].abs().sort_values(ascending=False)
correct_rank = correct_avg["avg_force"].abs().sort_values(ascending=False)

pd.DataFrame({
    "rank": range(1, len(beta) + 1),
    "library ranking":  [f"{a}->{b}" for a, b in library_rank.index],
    "correct ranking":  [f"{a}->{b}" for a, b in correct_rank.index],
})

,rank,library ranking,correct ranking
0,1,TFB->G4,TFA->G1
1,2,TFA->G1,TFB->G4
2,3,TFB->G2,TFB->G2
3,4,TFB->G3,TFB->G3


In [17]:
def top_set(series, percentile):
    return {f"{a}->{b}" for a, b in
            series[np.abs(series) >= np.percentile(np.abs(series), percentile)].index}

for pct in (50, 75):
    lib = top_set(avg["avg_force"], pct)
    cor = top_set(correct_avg["avg_force"], pct)
    print(f"percentile={pct}:  library {sorted(lib)}   correct {sorted(cor)}   "
          f"{'same set' if lib == cor else '<-- DIFFERENT EDGES SELECTED'}")

print("\nselect_top_edges(50) on the object:",
      [f"{a}->{b}" for a, b in epi.select_top_edges(percentile=50).index])

percentile=50:  library ['TFA->G1', 'TFB->G4']   correct ['TFA->G1', 'TFB->G4']   same set
percentile=75:  library ['TFB->G4']   correct ['TFA->G1']   <-- DIFFERENT EDGES SELECTED

select_top_edges(50) on the object: ['TFA->G1', 'TFB->G4']


### 2.8 The proposed fix

Index by name instead of by position.  Defined here as a local function — the
library is not modified.

In [18]:
def fixed_force_curves_chunk(beta_chunk, tf_expression, epsilon=EPS):
    """calculate_force_curves_chunk with a name-based reindex."""
    expanded = tf_expression.reindex(beta_chunk.index.get_level_values(0)).values
    b = beta_chunk.to_numpy()
    return pd.DataFrame(
        np.sign(b) * np.exp(np.log10(np.abs(b) + epsilon) + np.log10(expanded + epsilon)),
        index=beta_chunk.index, columns=beta_chunk.columns,
    )

fixed = fixed_force_curves_chunk(beta, tf_expr).mean(axis=1)
check = pd.DataFrame({
    "edge": [f"{a}->{b_}" for a, b_ in beta.index],
    "library": avg["avg_force"].values,
    "fixed":   fixed.values,
    "correct": correct_avg["avg_force"].values,
})
check["fixed is correct"] = np.isclose(check["fixed"], check["correct"])
check.round(4)

,edge,library,fixed,correct,fixed is correct
0,TFA->G1,1.5909,1.8259,1.8259,True
1,TFB->G2,-1.4040,-1.4040,-1.4040,True
2,TFB->G3,1.0686,1.0686,1.0686,True
3,TFB->G4,-2.0117,-1.7528,-1.7528,True


In [19]:
# ... and it is a no-op on the mini example too
print(np.allclose(fixed_force_curves_chunk(mini_beta, mini_expr)["time_0"].values,
                  reference_force(mini_beta["time_0"].values, np.array([10.0, 1000.0, 1000.0]))))

True


### 2.10 The same failure in a real analysis notebook

Everything above goes through the `EpisodeDynamics` methods.  The repo's own
`multiome_dynamic_regulation/py_scripts/analysis/LF_local_dynamics.ipynb`
(cells 28–29) builds the TF-expression frame by hand and calls
`calculate_force_curves_parallel` directly, with the production defaults:

```python
# cell 28
tf_lcpm_values  = lcpm_dcurve.loc[filtered_edges_p001.index.get_level_values(0).unique()]
tf_lcpm_episode = tf_lcpm_values.iloc[:, 35:40]
tf_lcpm_episode.columns = beta_time_cols[:n_time_cols]

# cell 29
force_curves = calculate_force_curves_parallel(
    beta_curves=filtered_edges_p001.drop('p_value', axis=1),
    tf_expression=tf_lcpm_episode,
    n_processes=20, chunk_size=30000, epsilon=1e-10, save_intermediate=False)
```

`.unique()` gives first-appearance order — the row-group order, the same frame
`compute_tf_expression` builds — so this notebook is exposed to the same
mis-pairing.  (Its manual `iloc[:, 35:40]` is a hand-written workaround for
issue 3, which the class method does not do.)  Reproduced with those defaults:

In [20]:
# the LF_local_dynamics.ipynb idiom, on this episode, with its parameters
tf_lcpm_values  = lcpm.loc[kept.index.get_level_values(0).unique()]
beta_time_cols  = [c for c in kept.columns if c.startswith("time_")]
tf_lcpm_episode = tf_lcpm_values.iloc[:, 0:len(beta_time_cols)].copy()
tf_lcpm_episode.columns = beta_time_cols

force_real = calculate_force_curves_parallel(
    beta_curves=kept.drop("p_value", axis=1),
    tf_expression=tf_lcpm_episode,
    n_processes=20, chunk_size=30000, epsilon=1e-10, save_intermediate=False,
)

pd.DataFrame([
    {"edge": f"{tf}->{target}", "should use": tf,
     "actually used": next(
         (c for c in tf_lcpm_episode.index
          if np.allclose(implied_expression(force_real.loc[(tf, target)].values,
                                            kept.loc[(tf, target)].drop("p_value").values),
                         tf_lcpm_episode.loc[c].values, atol=1e-6)), "?")}
    for tf, target in kept.index
]).assign(correct=lambda d: d["should use"] == d["actually used"])

Processing 4 edges using 20 processes...
Chunk size: 30,000 rows
Time columns: ['time_0', 'time_1', 'time_2', 'time_3', 'time_4']
Beta curves shape: (4, 5)
TF expression shape: (2, 5)
Created 1 chunks
Chunk sizes: [4]...


Processing chunks: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 119.92it/s]

Processing completed in 0.24 seconds
Combining results...
Final shape: (4, 5)


,edge,should use,actually used,correct
0,TFA->G1,TFA,TFB,False
1,TFB->G2,TFB,TFB,True
2,TFB->G3,TFB,TFB,True
3,TFB->G4,TFB,TFA,False


**Is the skew contrived?**  No — the two orderings are structurally unrelated.
`dictys` builds `nids[0]` with `sorted(...)`, so `build_episode_grn` emits TF row
groups in **alphabetical** order, while `value_counts()` orders them by
**descending target count**.

In [21]:
import math
print("regulator rows are alphabetical (dictys builds nids[0] with sorted()):")
print("   ", [str(net_skewed.nname[i]) for i in net_skewed.nids[0]])
print("\nfor n TFs with distinct target counts, the orders coincide by chance with"
      " probability 1/n!:")
for n in (2, 5, 10, 50):
    print(f"    n={n:<3} -> {1 / math.factorial(n):.3g}")

regulator rows are alphabetical (dictys builds nids[0] with sorted()):
    ['TFA', 'TFB', 'ZNF1']

for n TFs with distinct target counts, the orders coincide by chance with probability 1/n!:
    n=2   -> 0.5
    n=5   -> 0.00833
    n=10  -> 2.76e-07
    n=50  -> 3.29e-65


### 2.9 `get_tf_indices` is not in this path

It was suggested that `utils.custom.get_tf_indices` might account for the
ordering.  It is called from exactly one place, and it is not this one:

In [22]:
# No shelling out, no filesystem paths: read the installed modules' own source.
import firefate.core.episodic_dynamics as ed_mod
import firefate.core.pseudotime_curves as pc_mod

for mod in (ed_mod, pc_mod):
    refs = [f"  line {i}: {l.strip()}"
            for i, l in enumerate(inspect.getsource(mod).splitlines(), 1)
            if "get_tf_indices" in l]
    print(f"{mod.__name__}: {len(refs)} reference(s)")
    print("\n".join(refs) if refs else "  (none)")

print("\nget_tf_indices iterates the list it is given, so it preserves that order:")
print(inspect.getsource(get_tf_indices).split("for gene in tf_list:")[1].split("return")[0])

firefate.core.episodic_dynamics: 0 reference(s)
  (none)
firefate.core.pseudotime_curves: 1 reference(s)
  line 206: TF_indices, _, missing_tfs = get_tf_indices(self.dictys_dynamic_object, tf_list)

get_tf_indices iterates the list it is given, so it preserves that order:

        # Check if the gene is in the gene_hashmap
        if gene in gene_hashmap:
            gene_index = gene_hashmap[gene]  # Get the index in gene_hashmap
            # Check if the gene index is present in tf_mappings_to_gene_hashmap
            match = np.where(tf_mappings_to_gene_hashmap == gene_index)[0]
            if match.size > 0:  # If a match is found
                tf_indices.append(int(match[0]))  # Append the position of the match
                tf_gene_indices.append(int(gene_index))  # Also append the gene index
            else:
                missing_tfs.append(gene)  # Gene exists but not as a TF
        else:
            missing_tfs.append(gene)  # Gene not found at all
    


---
## 3. Issue 2 — the static `calculate_force_curves`

**What issue 2 is:** `SmoothedCurvesGRN.calculate_force_curves` decides which
expression row goes with which beta row **by position**, never by TF name, and
never checks that the caller's ordering matches.  Pandas has label-based
alignment built in (`.reindex`); this function does not use it.

**Note: Issue 2 is not about the cross product(issue 13).  That `get_beta_curves` returns
`T x G` rows instead of the requested links is **issue 13**, a separate finding.

The two appear together below because they are coupled: the cross product is
exactly what keeps issue 2 harmless today.  Every TF in a cross product has the
same number of targets, so `value_counts()` ties, ties keep first-appearance
order, and the positional pairing happens to land correctly.

> **Consequence worth flagging:** fixing issue 13 alone — returning only the
> requested links — would *activate* issue 2.  A filtered frame has unequal
> target counts per TF, `value_counts()` then re-sorts, and the positional
> pairing breaks.  Asking for `[(TFA,G1), (TFB,G4), (TFB,G5)]` and passing the
> expression in group order gives `TFB->G4` TFA's expression (10 instead of
> 1000).  The two should be fixed together.

### 3.1 Every caller in the repo already aligns by name

| caller | expression frame it passes |
|---|---|
| `state_dynamics.py` · `compute_forces` | `tf_expression.loc[beta_curves.index.get_level_values(0).unique()]` |
| `state_dynamics.py` · `force_curves_for_links` | same |
| `LF_global_dynamics.ipynb` cell 29 | same |
| `t_cell_analysis.ipynb` cell 21 | same (cell 26 goes further, a full per-row `reindex`) |

`.unique()` returns first-appearance order — the beta frame's own TF-group
order, which is what the positional `np.repeat` needs.  **No current caller
produces wrong forces through this function.**

In [23]:
# get_beta_curves returns a full cross product, so every TF has the same number
# of targets; with tied counts value_counts() preserves first-appearance order.
curves2 = SmoothedCurvesGRN(net_standard, trajectory_range=(0, 2),
                            num_points=6, dist=0.3, sparsity=0.1)
beta2, dtime2 = curves2.get_beta_curves([("TFA", "G1"), ("TFB", "G4")], varname="w")
tf_expr2, _ = curves2.get_smoothed_curves(mode="tf_expression")
tf_expr2.columns = beta2.columns

print("requested 2 links, got rows:", [f"{a}->{b}" for a, b in beta2.index],
      " <- cross product (issue 13)")
print("targets per TF            :", dict(beta2.index.get_level_values(0).value_counts()))
print("all counts equal          :",
      beta2.index.get_level_values(0).value_counts().nunique() == 1)

def audit(beta_frame, expr_frame, order):
    """Which edges come out of calculate_force_curves correctly paired?"""
    out = SmoothedCurvesGRN.calculate_force_curves(beta_frame, expr_frame.loc[order])
    return pd.DataFrame([
        {"edge": f"{tf}->{target}",
         "beta is zero": bool(np.all(beta_frame.loc[(tf, target)].values == 0)),
         "correct": bool(np.allclose(
             out.loc[(tf, target)].values,
             reference_force(beta_frame.loc[(tf, target)].values,
                             expr_frame.loc[tf].values)))}
        for tf, target in beta_frame.index])

# the idiom every real caller uses
regulon_tf_expression = tf_expr2.loc[beta2.index.get_level_values(0).unique()]
print("\nrepo idiom  .loc[beta.index.get_level_values(0).unique()]:")
audit(beta2, tf_expr2, list(regulon_tf_expression.index))

requested 2 links, got rows: ['TFB->G1', 'TFB->G4', 'TFA->G1', 'TFA->G4']  <- cross product (issue 13)
targets per TF            : {'TFB': np.int64(2), 'TFA': np.int64(2)}
all counts equal          : True

repo idiom  .loc[beta.index.get_level_values(0).unique()]:


,edge,beta is zero,correct
0,TFB->G1,True,True
1,TFB->G4,False,True
2,TFA->G1,False,True
3,TFA->G4,True,True


In [24]:
# If issue 13 were "fixed" on its own -- return only the requested links -- the
# frame would have unequal target counts, and issue 2 would start biting.
subset_idx = pd.MultiIndex.from_tuples(
    [("TFA", "G1"), ("TFB", "G4"), ("TFB", "G5")], names=["TF", "Target"])
subset_beta = pd.DataFrame(1.0, index=subset_idx, columns=["time_0"])
subset_expr = pd.DataFrame({"time_0": [10.0, 1000.0]}, index=["TFA", "TFB"])  # group order

print("counts per TF:", dict(subset_idx.get_level_values(0).value_counts()),
      "-> unequal, so value_counts() re-sorts")
out = SmoothedCurvesGRN.calculate_force_curves(subset_beta, subset_expr)
pd.DataFrame({
    "edge": [f"{a}->{b}" for a, b in subset_idx],
    "expression used": implied_expression(out["time_0"].values,
                                          subset_beta["time_0"].values).round(1),
    "should be": [subset_expr.loc[tf, "time_0"] for tf, _ in subset_idx],
}).assign(correct=lambda d: np.isclose(d["expression used"], d["should be"]))

counts per TF: {'TFB': np.int64(2), 'TFA': np.int64(1)} -> unequal, so value_counts() re-sorts


,edge,expression used,should be,correct
0,TFA->G1,10.0,10.0,True
1,TFB->G4,10.0,1000.0,False
2,TFB->G5,1000.0,1000.0,True


### 3.2 The alignment is load-bearing but unenforced

Nothing inside `calculate_force_curves` checks that invariant.  The cell below
passes the expression in a different order to show what an unwary future caller
would get — no exception, no warning, just wrong numbers for every non-zero
edge.  **This is hypothetical: no current caller does it.**  It is here to show
why the convention matters and why a name-based reindex inside the function
would be cheap insurance.

In [25]:
groups = list(dict.fromkeys(beta2.index.get_level_values(0)))
print("expression supplied as", groups[::-1], "instead of", groups)
audit(beta2, tf_expr2, groups[::-1])

expression supplied as ['TFA', 'TFB'] instead of ['TFB', 'TFA']


,edge,beta is zero,correct
0,TFB->G1,True,True
1,TFB->G4,False,False
2,TFA->G1,False,False
3,TFA->G4,True,True


The two rows that survive are the ones whose beta is exactly zero, where
`sign(0) == 0` erases the expression regardless.

### 3.3 The slip that is actually plausible

Passing `get_smoothed_curves(mode="tf_expression")` straight through, without the
`.loc[...unique()]`.  That frame is in `nids[0]` (alphabetical) order, whereas
`get_beta_curves` derives its row order from `list(set(...))` — so the two
generally differ.

In [26]:
print("tf_expression rows (nids[0], alphabetical):", list(tf_expr2.index))
print("beta frame's TF-group order              :", groups)
print()
try:
    display(audit(beta2, tf_expr2, list(tf_expr2.index)))
except Exception as exc:
    print(f"{type(exc).__name__}: {exc}")
    print("\n(The raw frame carries a row for every regulator, including ones with no "
          "rows in the beta frame, so this particular slip raises rather than "
          "silently mis-pairing -- which is the good case.)")

tf_expression rows (nids[0], alphabetical): ['TFA', 'TFB', 'ZNF1']
beta frame's TF-group order              : ['TFB', 'TFA']

ValueError: operands could not be broadcast together with shape (3,) (2,)

(The raw frame carries a row for every regulator, including ones with no rows in the beta frame, so this particular slip raises rather than silently mis-pairing -- which is the good case.)


### 3.4 Why the required order cannot be predicted

`get_beta_curves` builds its row order with `list(set(...))`, so it is neither
the caller's link order nor stable between interpreter runs (issue 13).  That is
what makes "derive the order from the returned frame" the only safe idiom:

In [27]:
# The beta frame's row order is decided by one expression inside get_beta_curves.
# Confirm that in-process first:
links = [("TFA", "G1"), ("TFB", "G4"), ("ZNF1", "G2")]
curves3 = SmoothedCurvesGRN(net_standard, (0, 2), num_points=4, dist=0.3, sparsity=0.1)
beta3, _ = curves3.get_beta_curves(links, varname="w")

tf_list = list(set([link[0] for link in links]))        # <- the line in question
print("list(set(...)) in this process:", tf_list)
print("beta frame's TF-group order   :",
      list(dict.fromkeys(beta3.index.get_level_values(0))))
print("they agree                    :",
      tf_list == list(dict.fromkeys(beta3.index.get_level_values(0))))

# Now the same expression in fresh interpreters.  No imports at all, so this
# cell cannot fail for environment reasons.
child = textwrap.dedent("""
    links = [("TFA", "G1"), ("TFB", "G4"), ("ZNF1", "G2")]
    print(list(set([l[0] for l in links])), "x", list(set([l[1] for l in links])))
""")
print("\nsame expression, fresh interpreters (TFs x targets):")
for seed in (1, 7, 42, 99):
    out = subprocess.run([sys.executable, "-c", child], capture_output=True, text=True,
                         env=dict(os.environ, PYTHONHASHSEED=str(seed)))
    print(f"  PYTHONHASHSEED={seed:<3}", out.stdout.strip() or out.stderr.strip()[-200:])

list(set(...)) in this process: ['TFB', 'TFA', 'ZNF1']
beta frame's TF-group order   : ['TFB', 'TFA', 'ZNF1']
they agree                    : True

same expression, fresh interpreters (TFs x targets):
  PYTHONHASHSEED=1   ['TFB', 'TFA', 'ZNF1'] x ['G1', 'G4', 'G2']
  PYTHONHASHSEED=7   ['ZNF1', 'TFB', 'TFA'] x ['G1', 'G2', 'G4']
  PYTHONHASHSEED=42  ['TFA', 'TFB', 'ZNF1'] x ['G4', 'G2', 'G1']
  PYTHONHASHSEED=99  ['ZNF1', 'TFB', 'TFA'] x ['G2', 'G4', 'G1']


### 3.5 The case the `xfail` test pins — unequal target counts

Reachable only by hand-building or hand-filtering a beta frame, never by
`get_beta_curves`.  Note the static method differs from
`calculate_force_curves_chunk`: it does **not** reindex `tf_expression` at all,
so it uses the caller's row order for the expression *values* but the
count-sorted order for the repeat *counts*.  Those two wrongs cancel in some
layouts and not in others, which is why this is a footgun rather than a
consistent error.

In [28]:
# Built explicitly -- this is not something get_beta_curves can return.
hand_index = pd.MultiIndex.from_tuples(
    [("TFA", "G1"), ("TFB", "G2"), ("TFB", "G3")], names=["TF", "Target"])
hand_beta = pd.DataFrame(1.0, index=hand_index, columns=["time_0"])
hand_expr = pd.DataFrame({"time_0": [10.0, 1000.0]}, index=["TFA", "TFB"])

print("counts     :", dict(hand_index.get_level_values(0).value_counts()))
print("row order  :", list(dict.fromkeys(hand_index.get_level_values(0))))
print("count order:", list(hand_index.get_level_values(0).value_counts().index))

hand_out = SmoothedCurvesGRN.calculate_force_curves(hand_beta, hand_expr)
pd.DataFrame({
    "edge": [f"{a}->{b}" for a, b in hand_index],
    "expression used": implied_expression(hand_out["time_0"].values,
                                          hand_beta["time_0"].values).round(3),
    "should be": [hand_expr.loc[tf, "time_0"] for tf, _ in hand_index],
    "correct": np.isclose(implied_expression(hand_out["time_0"].values,
                                             hand_beta["time_0"].values),
                          [hand_expr.loc[tf, "time_0"] for tf, _ in hand_index]),
})

counts     : {'TFB': np.int64(2), 'TFA': np.int64(1)}
row order  : ['TFA', 'TFB']
count order: ['TFB', 'TFA']


,edge,expression used,should be,correct
0,TFA->G1,10.0,10.0,True
1,TFB->G2,10.0,1000.0,False
2,TFB->G3,1000.0,1000.0,True


---
## 4. Summary

In [29]:
n_wrong = int((verdict["verdict"] == "WRONG").sum())
pd.DataFrame([
    {"issue": 1,
     "function": "calculate_force_curves_chunk",
     "reached by": "EnrichmentManager.enrich_episodic / run_episodic_construction / "
                   "LF_local_dynamics.ipynb",
     "trigger": "unequal target counts after filter_edges (the normal case)",
     "evidence": f"{n_wrong}/{len(verdict)} edges scaled by the wrong TF; "
                 f"top-edge selection changes at percentile 75",
     "status": "CONFIRMED -- active in production code"},
    {"issue": 2,
     "function": "SmoothedCurvesGRN.calculate_force_curves",
     "reached by": "state_dynamics.py, LF_global_dynamics.ipynb, t_cell_analysis.ipynb",
     "trigger": "expression rows not in the beta frame's TF-group order",
     "evidence": "all four real call sites use .loc[...unique()] and are correct; "
                 "the invariant is simply unenforced",
     "status": "REVISED -- latent, no caller currently affected"},
])

,issue,function,reached by,trigger,evidence,status
0,1,calculate_force_curves_chunk,EnrichmentManager.enrich_episodic / run_episod...,unequal target counts after filter_edges (the ...,2/4 edges scaled by the wrong TF; top-edge sel...,CONFIRMED -- active in production code
1,2,SmoothedCurvesGRN.calculate_force_curves,"state_dynamics.py, LF_global_dynamics.ipynb, t...",expression rows not in the beta frame's TF-gro...,all four real call sites use .loc[...unique()]...,"REVISED -- latent, no caller currently affected"


In [30]:
assert n_wrong == 2, "issue 1 no longer reproduces -- was calculate_force_curves_chunk fixed?"
assert audit(beta2, tf_expr2, list(regulon_tf_expression.index))["correct"].all(), \
    "the repo's .loc[...unique()] idiom no longer gives correct forces -- investigate"
assert not audit(beta2, tf_expr2, groups[::-1])["correct"].all(), \
    "calculate_force_curves now aligns by name -- issue 2 can be closed"
print("Issue 1 reproduces; issue 2's invariant holds for the repo's idiom and fails "
      "outside it. If any assertion fires, the library changed -> update ISSUES.md.")

Issue 1 reproduces; issue 2's invariant holds for the repo's idiom and fails outside it. If any assertion fires, the library changed -> update ISSUES.md.
